# Llama 3.2 Fine-Tuning for Personalized Mental Wellness Assistant

This notebook handles fine-tuning of `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` (or `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`) on the `ShenLab/MentalChat16K` dataset using Unsloth and Low-Rank Adaptation (LoRA).


In [ ]:
# 1. Install Unsloth and training dependencies
!pip install unsloth
!pip install --no-deps "trl<0.13.0"
!pip install bitsandbytes datasets accelerate peft huggingface_hub


In [ ]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face Hub
hf_token = os.environ.get("HF_TOKEN", "your_huggingface_write_token")
if hf_token != "your_huggingface_write_token":
    login(token=hf_token)


# Model Initialization and LoRA Adapter Configuration

Loads `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` with 4-bit quantization and injects trainable rank-16 LoRA adapters.


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("Llama 3.2 model and LoRA adapters loaded successfully.")


# Dataset Loading & Llama 3 Chat Formatting

Preprocesses `ShenLab/MentalChat16K` into the official Llama 3 instruction prompt template:

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a warm, empathetic mental health companion for the Personalized Mental Wellness Assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>
```


In [ ]:
from datasets import load_dataset

# Load MentalChat16K dataset
dataset = load_dataset("ShenLab/MentalChat16K", split="train")

llama3_prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a warm, supportive, and empathetic mental health companion for the Personalized Mental Wellness Assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

def format_llama3_prompts(examples):
    instructions = examples.get("instruction", [""] * len(examples["output"]))
    user_inputs = examples.get("input", [""] * len(examples["output"]))
    outputs = examples.get("output", [""] * len(examples["output"]))

    texts = []
    for inst, user_in, output in zip(instructions, user_inputs, outputs):
        full_input = f"{inst} {user_in}".strip()
        text = llama3_prompt_style.format(input=full_input, output=output)
        texts.append(text)
    return {"text": texts}

# Map formatting and split (95% Train, 5% Test)
dataset = dataset.map(format_llama3_prompts, batched=True)
dataset = dataset.train_test_split(test_size=0.05, seed=3407)

print(f"Data formatting complete. Train count: {len(dataset['train'])} | Eval count: {len(dataset['test'])}")


In [ ]:
print("Sample Formatted Prompt:")
print(dataset['train'][0]['text'])


# Supervised Fine-Tuning Execution

Executes `SFTTrainer` with gradient accumulation, mixed-precision (BF16/FP16), and step-based Hub checkpoint synchronization.


In [ ]:
import os
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported
from huggingface_hub import snapshot_download

repo_id = "A5CENSION-SRT/mental-health-llama3-v1"
checkpoint_to_resume = "checkpoint-500"
local_checkpoint_path = "./hf_checkpoint_cache"

try:
    snapshot_download(
        repo_id=repo_id,
        allow_patterns=f"{checkpoint_to_resume}/*",
        local_dir=local_checkpoint_path,
        token=hf_token
    )
    resume_from_dir = os.path.join(local_checkpoint_path, checkpoint_to_resume)
    print(f"Resuming training from cached checkpoint: {resume_from_dir}")
except Exception as e:
    print(f"No existing remote checkpoint found ({e}). Starting fresh training.")
    resume_from_dir = None

sft_config = SFTConfig(
    output_dir="./outputs_llama3",
    dataset_text_field="text",
    max_seq_length=1024,
    packing=True,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    max_steps=1800,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    report_to="none",
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_token=hf_token,
    hub_strategy="all_checkpoints",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=sft_config,
)

print("Starting Llama 3.2 fine-tuning...")
trainer.train(resume_from_checkpoint=resume_from_dir if resume_from_dir and os.path.exists(resume_from_dir) else None)

print(f"Fine-tuning completed. Model artifacts pushed to: https://huggingface.co/{repo_id}")
